## Prepare the dataset for clustering analysis (objective 3.2)


## 1. Setup and Load Data

In [1]:
# ## 1) Import libraries
import pandas as pd 
from sklearn.preprocessing import StandardScaler
import numpy as np
import os
from config import (
    FEATURE_NAMES_PATH,
    HS_COLS,
    INPUT_DATA_PATH,
    OUTPUT_DIR,
    RAW_PATH,
    REVIEW_COLS,
    SCALED_PATH,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# LOAD DATA
df = pd.read_csv(INPUT_DATA_PATH)

## 2. Select Variables

Retains `patient_id`, healthcare utilisation counts (`HS_COLS`), and structured review date columns (`REVIEW_COLS`). Columns absent from the raw dataset are silently dropped.

In [2]:
keep_cols = ["patient_id"] + HS_COLS + REVIEW_COLS
available_cols = [c for c in keep_cols if c in df.columns]
print(f" Found {len(available_cols)} variables to keep.")

df = df[available_cols]

 Found 20 variables to keep.


## 3. Data Quality Check


In [3]:
summary = {
    "shape": df.shape,
    "missing": df.isna().sum().sort_values(ascending=False),
    "dtypes": df.dtypes,
    "numeric": df.describe().T,
}
print(summary)

{'shape': (1165, 20), 'missing': copd_review_date                      1164
asthma_review_date                    1162
med_review_date                       1142
ed_attendances_pre_0_3m                  0
prescriptions_pre_9_12m                  0
prescriptions_pre_6_9m                   0
prescriptions_pre_3_6m                   0
prescriptions_pre_0_3m                   0
hospital_admissions_pre_9_12m            0
hospital_admissions_pre_6_9m             0
patient_id                               0
hospital_admissions_pre_0_3m             0
primary_care_attendances_pre_9_12m       0
primary_care_attendances_pre_6_9m        0
primary_care_attendances_pre_3_6m        0
primary_care_attendances_pre_0_3m        0
ed_attendances_pre_9_12m                 0
ed_attendances_pre_6_9m                  0
ed_attendances_pre_3_6m                  0
hospital_admissions_pre_3_6m             0
dtype: int64, 'dtypes': patient_id                             int64
ed_attendances_pre_0_3m               

## 4. Create Binary Review Indicators


In [4]:

for review_col in REVIEW_COLS:
    
    binary_col = review_col.replace("_date", "_bin")
    df[binary_col] = np.where(df[review_col].notna(), 1, 0)

df.drop(columns=REVIEW_COLS, inplace=True, errors="ignore")


In [5]:
display(df.loc[:, df.columns.str.contains("_bin")].describe().T)
for col in df.filter(like="_bin").columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


,count,mean,std,min,25%,50%,75%,max
asthma_review_bin,1165.0,0.002575,0.050702,0.0,0.0,0.0,0.0,1.0
copd_review_bin,1165.0,0.000858,0.029298,0.0,0.0,0.0,0.0,1.0
med_review_bin,1165.0,0.019742,0.139174,0.0,0.0,0.0,0.0,1.0



--- asthma_review_bin ---
asthma_review_bin
0    1162
1       3
Name: count, dtype: int64

--- copd_review_bin ---
copd_review_bin
0    1164
1       1
Name: count, dtype: int64

--- med_review_bin ---
med_review_bin
0    1142
1      23
Name: count, dtype: int64


## 5. Scale and Save Data

In [6]:
binary_review_cols = [c for c in df.columns if c.endswith("_bin")]

# Scale 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[HS_COLS])
scaled_df = pd.DataFrame(X_scaled, columns=HS_COLS, index=df.index)

# Combine scaled HS vars + binary reviews
df_scaled = pd.concat([df[["patient_id"]], scaled_df, df[binary_review_cols]], axis=1)


df.to_csv(RAW_PATH, index=False, compression="gzip")
df_scaled.to_csv(SCALED_PATH, index=False, compression="gzip")

print("Saved processed clustering datasets:")
print(f"   • Raw file:    {RAW_PATH}")
print(f"   • Scaled file: {SCALED_PATH}")
print("Data preparation complete.")

Saved processed clustering datasets:
   • Raw file:    /Users/cu20932/Documents/Work/reducehf/output/clustering/real/clustering_raw.csv.gz
   • Scaled file: /Users/cu20932/Documents/Work/reducehf/output/clustering/real/clustering_scaled.csv.gz
Data preparation complete.
